<a href="https://colab.research.google.com/github/marcory-hub/yolo11n-on-grove-vision-ai-v2/blob/main/pt_to_int8_vela_tflite_2026_02_23.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# YOLO11n pt to int8.tflite for Grove Vision AI V2


Last accessed: 2026-02-22

- updated calibrationset codeblock (list for classes, skip labels)


Check if you have to replace the default options with your custom model name and image size:
- `imgsz`=192 (default), 224 (max) for Grove vision AI V2, or other sizes. Train the yolo model with the imgsz that you are going to use on the Grove Vision AI V2.)
- `model_name`=best.pt
- `nc` = 4 (or your number of classes, fe coco dataset has 80 classes)
- `class_names`='amel', 'vcra', 'vespsp', 'vvel' (or your default classnames, in this format)
- `name`=best (or your custom name)

Steps 1-4 you already did in the `YOLO11n Training on Google Colab` part of this repository.

1. Make sure images and labels from your dataset have this folder structure with these exact names. And add `data.yaml` to main folder.

```
🗂️ dataset
  🗂️ train
    🗂️ images
    🗂️ labels
  🗂️ valid
    🗂️ images
    🗂️ labels
  data.yaml
```

2. Zip the dataset folder to a file names `dataset.zip`.

3. Copy the `dataset.zip` file to the root `/content/drive/MyDrive/` of google drive, it is needed to make a callibration image set.

4. Copy the `dataset.zip` file also to `/content/drive/MyDrive/`.

5. In the `YOLO11n Training on Google Colab` you downloaded the result of the training. Unzip the file and in the folder `content/{project}/{run}/weights` you find the file `best.pt`. Copy this file also to `/content/drive/MyDrive/`.

## Data preparation

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# Copy zipped dataset to colab and unzip the dataset
!cp '/content/drive/MyDrive/dataset.zip' '/content/dataset.zip'
!unzip '/content/dataset.zip' -d '/content/dataset/'

# Copy best.pt
!cp '/content/drive/MyDrive/best.pt' '/content/best.pt'

In [ ]:
# Make calibrationset
import os
import random
import shutil
import yaml

nc = "4"
class_names = "'amel', 'vcra', 'vespsp', 'vvel'"
num_images = 500
original_train_dir = "/content/dataset/train"
original_valid_dir = "/content/dataset/valid"
temp_dir = "/content/temp_subset"

def copy_random_images(src_dir, dst_dir, num_images):
    images = [f for f in os.listdir(f"{src_dir}/images")
              if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    selected = random.sample(images, min(num_images, len(images)))
    os.makedirs(f"{dst_dir}/images", exist_ok=True)
    os.makedirs(f"{dst_dir}/labels", exist_ok=True)
    for img in selected:
        label = img.rsplit('.', 1)[0] + '.txt'
        shutil.copy(f"{src_dir}/images/{img}", f"{dst_dir}/images/{img}")
        if os.path.exists(f"{src_dir}/labels/{label}"):
            shutil.copy(f"{src_dir}/labels/{label}", f"{dst_dir}/labels/{label}")

copy_random_images(original_train_dir, f"{temp_dir}/train", num_images)
copy_random_images(original_valid_dir, f"{temp_dir}/valid", num_images)

nc_int = int(nc)
names_list = [name.strip().strip("'") for name in class_names.split(',')]

temp_data_string = f"""
train: /content/temp_subset/train/images
val: /content/temp_subset/valid/images
nc: {nc_int}
names: {names_list}
"""

with open("/content/temp_data.yaml", 'w') as f:
    f.write(temp_data_string.strip())

print("Created temp_data.yaml with train and valid images for calibration.")

# Install ultralytics and export best.pt to int8.tflite

In [ ]:
# Install ultralytics
!pip install -U ultralytics tensorflow

In [ ]:
import os
import random
import shutil
from ultralytics import YOLO
import yaml

nc = 4
class_names = ["amel", "vcra", "vespsp", "vvel"]
num_images_per_split = 500
train_dir = "/content/dataset/train"
valid_dir = "/content/dataset/valid"
temp_dir = "/content/temp_calibration"

os.makedirs(f"{temp_dir}/train/images", exist_ok=True)
os.makedirs(f"{temp_dir}/train/labels", exist_ok=True)
os.makedirs(f"{temp_dir}/valid/images", exist_ok=True)
os.makedirs(f"{temp_dir}/valid/labels", exist_ok=True)

def copy_random_images(src_dir, dst_dir, num_images):
    images = [f for f in os.listdir(f"{src_dir}/images") if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    selected = random.sample(images, min(num_images, len(images)))
    for img in selected:
        label = img.rsplit('.', 1)[0] + '.txt'
        shutil.copy(f"{src_dir}/images/{img}", f"{dst_dir}/images/{img}")
        if os.path.exists(f"{src_dir}/labels/{label}"):
            shutil.copy(f"{src_dir}/labels/{label}", f"{dst_dir}/labels/{label}")

copy_random_images(train_dir, f"{temp_dir}/train", num_images_per_split)
copy_random_images(valid_dir, f"{temp_dir}/valid", num_images_per_split)

yaml_dict = {
    "train": f"{temp_dir}/train/images",
    "val": f"{temp_dir}/valid/images",
    "nc": nc,
    "names": class_names
}

yaml_path = "/content/temp_calibration_data.yaml"
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_dict, f)

print(f"Calibration YAML created at {yaml_path}")

In [ ]:
from ultralytics import YOLO

# Make sure best.pt exists in /content/
model = YOLO("/content/best.pt")

# Export INT8 TFLite (with calibration data)
model.export(format="tflite", int8=True, data="/content/temp_calibration_data.yaml")

# The TFLite file will be saved in the same folder as your .pt model:
tflite_path = "/content/best.tflite"
print("Exported TFLite INT8 model at:", tflite_path)


# Zip and download tflite versions

In [ ]:
import shutil
from google.colab import files

# --- Paths ---
source_folder = "/content/best_saved_model"
zip_file = "/content/best_saved_model.zip"

# --- Create ZIP ---
shutil.make_archive(zip_file.replace('.zip',''), 'zip', source_folder)
print(f"Folder zipped to {zip_file}")

# --- Download to local machine ---
files.download(zip_file)

# Int8.tflite to int8_vela.tflite


In [ ]:
!pip install ethos-u-vela

In [ ]:
!vela /content/best_saved_model/best_full_integer_quant.tflite


In [ ]:
import shutil
from google.colab import files

# --- Paths ---
source_folder = "/content/output"
zip_file = "/content/output.zip"

# --- Create ZIP ---
shutil.make_archive(zip_file.replace('.zip',''), 'zip', source_folder)
print(f"Folder zipped to {zip_file}")

# --- Download to local machine ---
files.download(zip_file)

Prune if needed

In [ ]:
import torch
import torch.nn.utils.prune as prune

# Load YOLOv11n model
from ultralytics import YOLO
model = YOLO("best.pt")

# Example: prune all Conv2d layers
amount = 0.3  # remove 30% of channels

for module in model.model.modules():
    if isinstance(module, torch.nn.Conv2d):
        # structured pruning: remove output channels (dim=0)
        prune.ln_structured(module, name="weight", amount=amount, n=2, dim=0)
        prune.remove(module, "weight")  # make permanent

# Save pruned model
model.save("best_pruned.pt")

In [ ]:
from ultralytics import YOLO

# Load your model
model = YOLO("/content/best.pt")

# Customize validation settings
metrics = model.val(data="/content/dataset/data.yaml", imgsz=192, batch=16, conf=0.25, iou=0.7)

In [ ]:
from ultralytics import YOLO

# Load a model
model = YOLO("/content/best_pruned.pt")

# Customize validation settings
metrics = model.val(data="/content/dataset/data.yaml", imgsz=192)

---
# Himax

In [ ]:
from ultralytics import YOLO

# # Load a model
image_size = 192

model = YOLO("best.pt")

model.export(format="tflite", imgsz = image_size, int8 = True, data="/content/dataset/data.yaml")


In [ ]:
!pip install ethos-u-vela

In [ ]:
!vela --accelerator-config ethos-u55-64 \
      --output-dir ./best_saved_model \
      best_saved_model/best_int8.tflite